In [2]:
import tnved
from lxml import etree
import pandas as pd
import pymorphy3
from tqdm.notebook import tqdm
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re

In [3]:
tree=etree.parse('c:/проект/tnved.xml')
root=tree.getroot()
items=root.findall('.//item')
morph = pymorphy3.MorphAnalyzer()
def lemmatize_sentence(text):
    stop_words = set(stopwords.words('russian'))
    text_cleaned = re.sub(r'[^\w\s]', '', text)
    words = word_tokenize(text_cleaned, language='russian')
    lemmas = [morph.parse(word)[0].normal_form for word in words]
    return ' '.join([i for i in lemmas if not i in stop_words])
dict={(i.attrib['code'])[:4]:lemmatize_sentence(i.attrib['name']) for i in items if ('code' in i.attrib.keys()) and re.match(r'^\d{4}[0\s]*$',i.attrib['code'])}

In [ ]:
df=pd.read_parquet('dataset.parquet')
del df['z']
bad=['7907']
df=df[~(df['y'] in bad)]
tqdm.pandas(desc="Обработка")
df['target']=df['y'].progress_apply(lambda x:x[-2:])
df['y']=df['y'].progress_apply(lambda x:dict[x[:4]])
df.head()